# Module 07 — Chi-square & Non-parametric Tests
Dedicated notebook for Module 07 — reproduces all 6 tests in the lesson: chi-square (2 tables), Mann-Whitney U, Wilcoxon, Kruskal-Wallis, Friedman.

Human-reviewed English translation of the original Vietnamese notebook (code and results are identical, only text is translated).

SPSS equivalents: `Analyze > Descriptive Statistics > Crosstabs` (Chi-square), `Analyze > Nonparametric Tests > Legacy Dialogs`.

In [ ]:
# Run this cell first (Colab: click ▶). No installation needed.
import numpy as np, pandas as pd, matplotlib.pyplot as plt, seaborn as sns
from scipy import stats
sns.set_theme(style="whitegrid"); plt.rcParams["figure.figsize"] = (7, 4)
pd.set_option("display.precision", 3)

def make_data(seed=2026, n=240):
    """SIMULATED 'Classroom of 240 students' dataset — SAME seed=2026 used
    throughout Modules 03-08, exactly matching lop_hoc_240.csv/.sav in the
    repo (not real data, but 100% reproducible)."""
    rng = np.random.default_rng(seed)
    school = rng.choice(list("ABC"), n, p=[.35, .35, .30])
    gender = rng.choice(["Male", "Female"], n)
    method = rng.choice(["Traditional", "Project-based"], n)
    study_hours = np.clip(rng.gamma(4, 1.2, n), 0.5, 15).round(1)
    interest = rng.normal(0, 1, n) + 0.15 * (method == "Project-based")
    anxiety = rng.normal(0, 1, n) - 0.25 * interest
    def likert(lat, load, noise=0.7):
        return np.clip(np.round(3 + load * lat + rng.normal(0, noise, n)), 1, 5).astype(int)
    d = pd.DataFrame({"id": np.arange(1, n + 1), "school": school, "gender": gender, "method": method,
                      "study_hours": study_hours})
    for k, (lat, load) in enumerate([(interest, .8), (interest, .7), (interest, .75)], 1):
        d[f"h{k}"] = likert(lat, load)
    for k, (lat, load) in enumerate([(anxiety, .8), (anxiety, .75), (anxiety, .7)], 1):
        d[f"a{k}"] = likert(lat, load)
    eff = d.school.map({"A": 3, "B": 0, "C": -3}).to_numpy()
    d["pretest"] = (rng.normal(60, 10, n) + eff).round(1)
    d["posttest"] = np.clip(d.pretest + 3 + 5 * (method == "Project-based") + rng.normal(0, 6, n), 0, 100).round(1)
    d["math"] = np.clip(35 + 2.5 * study_hours + 4 * interest - 3 * anxiety + eff + rng.normal(0, 6, n), 0, 100).round(1)
    d.loc[6, "study_hours"] = 48.0
    d.loc[[11, 57, 130], "h2"] = np.nan
    d["h2"] = d["h2"].astype("Int64")
    return d

df = make_data()
print("Created df:", df.shape)

## 1. Chi-square: gender × teaching method (2×2 table, with Yates continuity correction)

🎯 **What question does this method answer?** Chi-square tests: are two NOMINAL variables INDEPENDENT of each other — in other words, does knowing the value of one variable help predict the other BETTER than random guessing? Null hypothesis (H0): the two variables are independent, meaning the OBSERVED frequency in each crosstab cell matches the EXPECTED frequency if there were no relationship between them at all.

In [ ]:
ct1 = pd.crosstab(df["gender"], df["method"])
display(ct1)
chi2, p, dof, exp = stats.chi2_contingency(ct1, correction=True)
print(f"chi2({dof})={chi2:.2f}, p={p:.3f}  (Yates continuity correction applied for the 2x2 table)")

#### 📤 Actual output
χ²(1)=1.37, p=0.243 → NOT significant. ✅ Matches the lesson.

## 2. Chi-square: gender × school (2×3 table) + Cramer's V

🎯 **What question does this method answer?** Cramer's V measures the STRENGTH of association between two nominal variables (playing a role similar to |r| in Pearson correlation) — ranging FROM 0 (no association) TO 1 (perfect association). Unlike the p-value (which only says statistically significant/not), Cramer's V tells you HOW STRONG that association is — two different questions, echoing the Module 04 lesson (statistical significance ≠ practical size).

In [ ]:
ct2 = pd.crosstab(df["gender"], df["school"])
display(ct2)
chi2b, pb, dofb, expb = stats.chi2_contingency(ct2, correction=False)
n_total = ct2.values.sum()
cramer_v = np.sqrt(chi2b / (n_total * (min(ct2.shape)-1)))
pct_small = (expb < 5).mean() * 100
print(f"chi2({dofb})={chi2b:.2f}, p={pb:.3f}, Cramer's V={cramer_v:.3f}")
print(f"% of cells with expected count <5: {pct_small:.0f}% (safety rule: must be <20%)")

#### 📤 Actual output
χ²(2)=0.14, p=0.932, Cramer's V≈0.024 (nearly 0), 0% of cells violate the 20% rule → the result is trustworthy, not significant.

## 3. Mann-Whitney U: `h1` rating by gender

🎯 **What question does this method answer?** Mann-Whitney U is the non-parametric counterpart to the independent t-test: it tests whether ONE of the two groups tends to produce HIGHER values than the other — based on the RANKS of all observations (both groups combined), not on means. Used when data is ordinal (like a Likert scale) or violates the t-test's normality assumption.

In [ ]:
h1_male = df.loc[df["gender"]=="Male", "h1"].dropna()
h1_female = df.loc[df["gender"]=="Female", "h1"].dropna()
u, p = stats.mannwhitneyu(h1_male, h1_female)
print(f"U={u:.1f}, p={p:.3f}")

#### 📤 Actual output
U=6416, p=0.127 → not significant.

## 4. Wilcoxon: comparing 2 rating items `a1` and `a2` (same 240 students)

🎯 **What question does this method answer?** Wilcoxon is the non-parametric counterpart to the paired t-test: it tests whether the DIFFERENCE between two measurements on the SAME subject tends to lean in one direction (positive or negative) — based on the RANKS of the difference magnitudes, not their mean value.

In [ ]:
w_stat, p_w = stats.wilcoxon(df["a1"], df["a2"])
print(f"W={w_stat:.1f}, p={p_w:.3f}")
print(f"Mean a1={df['a1'].mean():.3f}, Mean a2={df['a2'].mean():.3f}")

#### 📤 Actual output
p=0.920 — no difference at all (mean a1≈2.908, a2≈2.904, very close).

## 5. Kruskal-Wallis: `h1` rating by school (3 groups)

🎯 **What question does this method answer?** Kruskal-Wallis is the non-parametric counterpart to one-way ANOVA: it tests whether 3+ INDEPENDENT groups have different RANK distributions — used when data is ordinal or violates ANOVA's normality assumption. Like ANOVA, a significant result does NOT identify which group differs.

In [ ]:
groups_h1 = [g["h1"].dropna().values for _, g in df.groupby("school")]
h_stat, p_kw = stats.kruskal(*groups_h1)
print(f"H={h_stat:.3f}, p={p_kw:.3f}")

#### 📤 Actual output
H=2.844, p=0.241 — not significant. Different from the significant result (F=5.87, p=0.003) when comparing Math scores by school in Module 06 — different variables, different answers.

## 6. Friedman: 3 rating items `a1`, `a2`, `a3` (same 240 students)

🎯 **What question does this method answer?** Friedman is the non-parametric counterpart to repeated-measures ANOVA: it tests whether 3+ measurements/variables on the SAME group of subjects tend to RANK differently (based on within-subject ranks, not raw mean values).

In [ ]:
chi2_f, p_f = stats.friedmanchisquare(df["a1"], df["a2"], df["a3"])
print(f"chi2={chi2_f:.3f}, p={p_f:.3f}")

#### 📤 Actual output
χ²=0.555, p=0.758 — not significant, students rate these related aspects fairly evenly.